# Week 4: Production RAG Architecture

**Lab companion to [week_04.md](../agenda/week_04.md)**

In this lab, you will:
1. Implement document chunking strategies
2. Use a real vector database (Qdrant)
3. Build RAG with citations
4. Handle metadata and filtering

In [ ]:
# Install dependencies
!pip install qdrant-client openai python-dotenv -q

In [ ]:
# Setup
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from dotenv import load_dotenv
import uuid

load_dotenv()
openai_client = OpenAI()
qdrant = QdrantClient(":memory:")  # In-memory for demo

print("✓ Ready!")

## 1. Document Chunking

Split large documents into smaller pieces for better retrieval.

In [ ]:
# Sample document
long_document = """
# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience. Unlike traditional programming where rules are explicitly coded, ML systems learn patterns from data.

## Types of Machine Learning

### Supervised Learning
In supervised learning, the algorithm learns from labeled training data. Each example in the training set consists of an input and the correct output. The algorithm learns to predict outputs for new inputs.

Common supervised learning tasks include:
- Classification: Predicting a category (spam/not spam)
- Regression: Predicting a continuous value (house prices)

### Unsupervised Learning
Unsupervised learning works with unlabeled data. The algorithm must find patterns and structure in the data on its own.

Common unsupervised learning tasks include:
- Clustering: Grouping similar data points
- Dimensionality reduction: Simplifying data while preserving information

### Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It receives rewards or penalties based on its actions and learns to maximize cumulative reward.

## Neural Networks

Neural networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes that process information.

Deep learning refers to neural networks with many layers, capable of learning hierarchical representations of data.
"""

print(f"Document length: {len(long_document)} characters")

In [ ]:
def chunk_by_characters(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap

    return [c for c in chunks if c]  # Remove empty chunks

# Test
char_chunks = chunk_by_characters(long_document, chunk_size=300, overlap=50)
print(f"Created {len(char_chunks)} chunks\n")

for i, chunk in enumerate(char_chunks[:3]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:150] + "...")
    print()

In [ ]:
def chunk_by_sentences(text: str, max_chunk_size: int = 500) -> list:
    """Split text into chunks at sentence boundaries."""
    import re

    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += " " + sentence if current_chunk else sentence
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Test
sent_chunks = chunk_by_sentences(long_document, max_chunk_size=400)
print(f"Created {len(sent_chunks)} chunks\n")

for i, chunk in enumerate(sent_chunks[:3]):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
def chunk_by_sections(text: str) -> list:
    """Split markdown text by headers."""
    import re

    # Split by markdown headers
    sections = re.split(r'\n(?=#{1,3}\s)', text)

    chunks = []
    for section in sections:
        section = section.strip()
        if section:
            # Extract header as title
            lines = section.split('\n')
            title = lines[0].lstrip('#').strip() if lines[0].startswith('#') else "Untitled"
            content = '\n'.join(lines[1:]).strip() if len(lines) > 1 else ""

            if content:
                chunks.append({
                    'title': title,
                    'content': content
                })

    return chunks

# Test
sections = chunk_by_sections(long_document)
print(f"Found {len(sections)} sections:\n")

for section in sections:
    print(f"📄 {section['title']}")
    print(f"   {section['content'][:100]}...")
    print()

## 2. Setting Up Qdrant Vector Database

In [ ]:
# Create a collection
COLLECTION_NAME = "documents"
EMBEDDING_DIM = 1536  # text-embedding-3-small

# Delete if exists
try:
    qdrant.delete_collection(COLLECTION_NAME)
except:
    pass

# Create collection
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE
    )
)

print(f"✓ Created collection: {COLLECTION_NAME}")

In [ ]:
def get_embedding(text: str) -> list:
    """Get embedding for text."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def add_documents(chunks: list, source: str):
    """Add document chunks to Qdrant."""
    points = []

    for i, chunk in enumerate(chunks):
        if isinstance(chunk, dict):
            text = chunk.get('content', '')
            title = chunk.get('title', 'Untitled')
        else:
            text = chunk
            title = f"Chunk {i+1}"

        embedding = get_embedding(text)

        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding,
            payload={
                "text": text,
                "title": title,
                "source": source,
                "chunk_index": i
            }
        ))

    qdrant.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )

    print(f"Added {len(points)} chunks from {source}")

# Add our document
sections = chunk_by_sections(long_document)
add_documents(sections, "ml_intro.md")

In [ ]:
def search(query: str, top_k: int = 3) -> list:
    """Search for relevant chunks."""
    query_embedding = get_embedding(query)

    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_embedding,
        limit=top_k
    )

    return [
        {
            "text": r.payload["text"],
            "title": r.payload["title"],
            "source": r.payload["source"],
            "score": r.score
        }
        for r in results
    ]

# Test search
results = search("What is supervised learning?")
for r in results:
    print(f"Score: {r['score']:.3f} | Section: {r['title']}")
    print(f"  {r['text'][:150]}...")
    print()

## 3. RAG with Citations

In [ ]:
def ask_with_citations(question: str) -> dict:
    """Answer question with source citations."""

    # Retrieve
    results = search(question, top_k=3)

    if not results:
        return {
            "answer": "No relevant information found.",
            "sources": []
        }

    # Build context with source markers
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(f"[Source {i}: {r['title']}]\n{r['text']}")

    context = "\n\n".join(context_parts)

    # Generate with citation instructions
    response = openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "system",
                "content": """Answer questions based on the provided sources.
Cite sources using [Source N] format.
If the sources don't contain the answer, say so."""
            },
            {
                "role": "user",
                "content": f"""Sources:
{context}

Question: {question}"""
            }
        ]
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [
            {"title": r["title"], "source": r["source"], "score": r["score"]}
            for r in results
        ]
    }

# Test
result = ask_with_citations("What are the main types of machine learning?")

print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s['title']} ({s['source']}) [score: {s['score']:.3f}]")

## 4. Complete RAG Pipeline

In [ ]:
class ProductionRAG:
    """Production-ready RAG system."""

    def __init__(self, collection_name: str = "prod_docs"):
        self.openai = OpenAI()
        self.qdrant = QdrantClient(":memory:")
        self.collection_name = collection_name
        self.embedding_dim = 1536

        # Create collection
        self.qdrant.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=self.embedding_dim,
                distance=Distance.COSINE
            )
        )

    def add_document(self, text: str, source: str, metadata: dict = None):
        """Add a document with chunking."""
        chunks = chunk_by_sentences(text, max_chunk_size=500)

        points = []
        for i, chunk in enumerate(chunks):
            embedding = self._embed(chunk)

            payload = {
                "text": chunk,
                "source": source,
                "chunk_index": i,
                **(metadata or {})
            }

            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload=payload
            ))

        self.qdrant.upsert(
            collection_name=self.collection_name,
            points=points
        )

        return len(points)

    def query(self, question: str, top_k: int = 3) -> dict:
        """Answer a question with RAG."""

        # Retrieve
        query_emb = self._embed(question)
        results = self.qdrant.search(
            collection_name=self.collection_name,
            query_vector=query_emb,
            limit=top_k
        )

        if not results:
            return {"answer": "No relevant information found.", "sources": []}

        # Build context
        context = "\n\n".join([
            f"[{i+1}] {r.payload['text']}"
            for i, r in enumerate(results)
        ])

        # Generate
        response = self.openai.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {
                    "role": "system",
                    "content": "Answer based on the provided context. Cite sources with [N]."
                },
                {
                    "role": "user",
                    "content": f"Context:\n{context}\n\nQuestion: {question}"
                }
            ]
        )

        return {
            "answer": response.choices[0].message.content,
            "sources": [
                {"text": r.payload["text"][:100], "source": r.payload["source"], "score": r.score}
                for r in results
            ]
        }

    def _embed(self, text: str) -> list:
        response = self.openai.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

# Create and test
rag = ProductionRAG()

# Add documents
rag.add_document(long_document, "ml_intro.md", {"topic": "machine_learning"})
rag.add_document(
    "Python is a versatile programming language. It has clear syntax and is great for beginners.",
    "python_intro.md",
    {"topic": "programming"}
)

print("✓ RAG system ready")

In [ ]:
# Test queries
questions = [
    "What is the difference between supervised and unsupervised learning?",
    "What is reinforcement learning?",
    "Tell me about neural networks"
]

for q in questions:
    print(f"Q: {q}")
    result = rag.query(q)
    print(f"A: {result['answer']}")
    print(f"Sources: {[s['source'] for s in result['sources']]}")
    print("-" * 50)

## Summary

You learned:
- ✅ Document chunking strategies
- ✅ Using Qdrant vector database
- ✅ RAG with source citations
- ✅ Building a complete RAG pipeline

**Next:** [Week 7 - Evaluations & Metrics](week_05_evaluations.ipynb)